# GLM-5.3-Flash 企业部署：8×H20 / FP8 / vLLM

按顺序执行，每个代码单元前说明用途、输出和使用 GPU 的时点。操作目标为**专用 Ubuntu 24.04 x86_64 服务器**，不是共享训练服务器。模型为 `zai-org/GLM-5.3-Flash`；API 名为 `glm-5.3-flash`。

- `/srv/phyxi` 必须是管理员预先挂载的数据卷；本 Notebook 不格式化磁盘。
- 推荐主机内存 ≥768 GiB（脚本门槛 600 GiB）、数据卷 ≥1.5 TB，首次制备至少 600 GiB 空闲。单份 FP8 权重约 306 GiB，实际文件量在下载后记录。
- 安装驱动可能要求重启，企业 CA 签发需要管理员提交 CSR。这两步有明确停点，完成后继续，不要忽略错误。
- 完成准备后，只有“启动服务”单元会初始化 GPU 并加载模型；本地静态验证不代表目标服务器已验收。
- 初始上下文 64K、调度序列上限 16 是可调整部署参数，**不是实测承载人数**。
- 图片请求使用 Python 前端；H20 使用 FP8 权重与 BF16 KV，非 FP8 KV。不开启 MTP/PD 分离，先完成基础服务验收。

运行前先按 README 安装并启动 Jupyter；此 Notebook 的管理员 kernel 仅监听回环，通过 SSH 隧道访问。
本文路径以默认 `/srv/phyxi` 展示；修改 `data_root` 后，生成的目录、挂载、脚本和 systemd 路径会同步替换。

## 部署架构与 OpenCode 请求链路

```mermaid
flowchart TB
    subgraph ADMIN["① 部署配置与服务管理"]
        NB["部署 Notebook<br/>单元格修改域名、IP、GPU UUID、版本和资源参数"]
        CONFIG["config.local.json<br/>执行配置单元后保存"]
        SCRIPT["scripts/deploy.py<br/>安装环境、检查版本、下载权重、生成配置"]
        RELEASE["/srv/phyxi/releases/发布编号/<br/>compose.yaml · .env · nginx.conf<br/>start-model.sh · prometheus.yml"]
        CURRENT["/srv/phyxi/current<br/>指向当前发布目录"]
        SYSTEMD["phyxi-model.service<br/>通过 Docker Compose 管理三个容器"]
        NB --> CONFIG --> SCRIPT --> RELEASE
        RELEASE --> CURRENT --> SYSTEMD
    end

    subgraph WS["② 企业 Windows 工作站"]
        SETUP["setup-workstation.ps1<br/>安装项目、连接适配和 OpenCode"]
        FILES["%LOCALAPPDATA% / PhyXiEnterprise /<br/>opencode-gateway/opencode.json<br/>opencode-gateway/connection.json<br/>company-ca.pem · cad-ws-01.token"]
        UI["PhyXiCATIAagent WebUI<br/>使用项目连接适配"]
        OC["独立 OpenCode 2.2.0<br/>监听本机 127.0.0.1 动态端口<br/>模型：phyxi-enterprise / glm-5.3-flash"]
        SETUP --> FILES
        FILES -. "connection.json：本机地址与密码" .-> UI
        FILES -. "opencode.json：网关地址与模型配置" .-> OC
        UI <-->|"本机 API · 会话与消息"| OC
    end

    subgraph SERVER["③ 专用推理服务器 · Ubuntu 24.04 · 8×H20"]
        ENTRY["企业模型域名 → 企业内网 IP:443"]
        subgraph DOCKER["Docker Compose · phyxi-enterprise · inference 桥接网络"]
            GW["Nginx 网关容器<br/>容器端口 8443<br/>TLS、来源网段、工作站 Token 校验<br/>连接数限制、流式响应转发"]
            VLLM["vLLM 模型容器 · model:8000<br/>OpenAI 兼容 API · Python 多模态前端<br/>glm47 工具调用与思考解析器"]
            PROM["Prometheus 容器<br/>采集推理指标、保存监控数据"]
        end
        GPU["8×H20 · TP=8<br/>一份 GLM-5.3-Flash 跨八卡推理<br/>原生 FP8 权重 · BF16 KV Cache"]
        MODEL["/srv/phyxi/models/glm53-flash/commit/<br/>固定 commit 的模型权重<br/>只读挂载到容器 /model"]
        CACHE["/srv/phyxi/cache/<br/>编译与运行缓存"]
        SECRET["/srv/phyxi/secrets/<br/>server.key · server-chain.pem · company-ca.pem<br/>client-keys.map · upstream-auth.conf<br/>backend-key · 工作站 Token"]
        MONDATA["/srv/phyxi/prometheus/<br/>持久化监控数据"]
        MONENTRY["服务器 127.0.0.1:9090<br/>管理员通过 SSH 隧道查看"]
        ENTRY -->|"宿主机 443 → 容器 8443"| GW
        GW <-->|"内部 HTTP · 替换为后端 API Key"| VLLM
        VLLM <--> GPU
        MODEL -. "读取权重" .-> VLLM
        CACHE -. "读写缓存" .-> VLLM
        SECRET -. "TLS、工作站认证和后端认证头" .-> GW
        SECRET -. "backend-key" .-> VLLM
        PROM -->|"每 15 秒抓取 /metrics"| VLLM
        PROM --> MONDATA
        MONENTRY --> PROM
    end

    EXPORT["/srv/phyxi/client-export/<br/>受控分发给指定工作站<br/>company-ca.pem · 工作站 Token<br/>endpoint.json 与客户端脚本"]
    SCRIPT -->|"生成密钥、CSR 并导入证书"| SECRET
    SCRIPT -->|"下载并校验"| MODEL
    SYSTEMD -->|"启动与停止"| DOCKER
    SCRIPT -->|"export 阶段生成分发材料"| EXPORT
    SECRET -->|"仅导出 CA 与工作站 Token"| EXPORT
    EXPORT --> FILES
    OC <-->|"HTTPS · 工作站 Token<br/>文字、图片、工具消息与 SSE 响应"| ENTRY

    classDef config fill:#eef2ff,stroke:#6366f1,color:#1e1b4b;
    classDef client fill:#ecfdf5,stroke:#10b981,color:#064e3b;
    classDef service fill:#eff6ff,stroke:#3b82f6,color:#1e3a8a;
    classDef storage fill:#fff7ed,stroke:#f59e0b,color:#78350f;
    class NB,CONFIG,SCRIPT,RELEASE,CURRENT,SYSTEMD config;
    class SETUP,FILES,UI,OC client;
    class ENTRY,GW,VLLM,PROM,GPU,MONENTRY service;
    class MODEL,CACHE,SECRET,MONDATA,EXPORT storage;
```

实线表示配置生成、服务管理或请求路径，虚线表示组件读取的配置、凭据与挂载文件。

- **配置生效**：修改 Notebook 并执行配置单元，再生成发布文件。运行中的部署先停止，变更已启动发布时使用新发布编号，随后启动并验收。
- **请求路径**：WebUI → 本机 OpenCode → Nginx → vLLM → 八张 H20；响应沿原路径返回。网关开放 `/v1/models` 和 `/v1/chat/completions`。
- **两层认证**：工作站 Token 由 `start-client.py` 读入 `PHYXI_GATEWAY_TOKEN` 环境变量，`opencode.json` 引用该变量。Nginx 验证后换成后端 API Key；后端 Key 和 TLS 私钥留在服务器。
- **端口与启动**：企业内网 IP 的 443 映射到网关 8443；模型 8000 不映射宿主机；监控 9090 只绑定服务器回环。网关等待模型健康后启动，Prometheus 没有该启动依赖。
- **工作站接入**：连接适配通过 `PHYXI_OPENCODE_CONNECTION` 读取 `connection.json`；OpenCode 通过 `OPENCODE_CONFIG` 读取自己的配置，使用 `NODE_EXTRA_CA_CERTS` 信任企业 CA。当前启动器默认禁止本地工具执行，业务工具按项目另行启用和验收。
- **目录**：图中 `/srv/phyxi` 为 `data_root` 默认值；首次修改后生成路径同步替换。`commit` 和“发布编号”代表实际模型 commit 与 `release` 值。Docker Engine 使用专用服务器的 `/var/run/docker.sock`，Docker 与 containerd 数据分别存放于数据根目录的 `docker/`、`containerd/`。


## 0. 定位仓库和运行环境

本单元不修改系统。将内核目录定位到仓库，后续动作由 `scripts/deploy.py` 的具名阶段执行；所有命令在脚本中可审阅。遇到任何命令失败即停止，不继续后续阶段。
`stage("docker")` 这类调用只是具名步骤入口，等价于 `python3 scripts/deploy.py docker --config config.local.json`。参数来自第 1 节，无需打开脚本修改。


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
REPO = Path.cwd().resolve()
if REPO.name == "notebooks": REPO = REPO.parent
assert (REPO / "scripts/deploy.py").is_file(), "请在本仓库内打开 Notebook"
assert os.geteuid() == 0, "需要按 README 启动管理员 Notebook 内核"
CONFIG_PATH = REPO / "config.local.json"  # 已加入 .gitignore；不要提交执行后的 Notebook

def stage(name):
    subprocess.run(["/usr/bin/python3", str(REPO / "scripts/deploy.py"), name,
                    "--config", str(CONFIG_PATH)], check=True)
print("Repository:", REPO)
print((REPO / "deployment/versions.json").read_text())

## 1. 集中修改与确认部署配置

**在下面的 `OVERRIDES` 单元格直接修改，不需要编辑 `.py`、`.sh`、JSON 或 YAML 文件。**

- `None` 表示保留上次保存值；首次运行时使用下表默认值。
- 要改某项，将 `None` 换成实际值：字符串加引号，数字不加引号，多个值用列表。例如 `"domain": "llm.company.internal"`。
- 首次必须填写域名、IP、允许网段，并确认专用服务器和数据卷。GPU UUID 可在驱动安装后的第 5 节获取，并直接在该节的 GPU_UUIDS 单元格填写保存；也可在本节填写。
- 执行本单元后写入被 Git 忽略的 `config.local.json`；`stage(...)` 每次都重新读取它。修改 Notebook 但未执行此单元，不会生效。
- 部署参数集中在本单元修改；GPU UUID 也可在第 5 节直接保存。首次生成后、尚未启动的配置可重新执行第 10 节生成；已经启动过的发布需先停止，在本单元改用新的 release 编号，再按第 10 节顺序重新生成。

| 字段 | 是否需要修改 | 默认值 | 获取方式 / 修改影响 |
|---|---|---|---|
| `dedicated_server` | 必须确认 | `true` | 确认是专用服务器；共享训练服务器不使用此 Notebook |
| `domain` | 必须填写 | `示例域名不可用` | 向企业网络管理员获取 DNS 域名，解析到 intranet_ip |
| `intranet_ip` | 必须填写 | `10.20.0.10 为示例` | 用 ip -br addr 查看服务器业务网卡地址 |
| `allowed_cidrs` | 必须填写 | `示例网段` | 由网络管理员给出工作站/应用出口网段；不是 GPU 网段 |
| `gpu_uuids` | 必须填写 | `空列表` | 第 5 节 nvidia-smi 获取八张已分配 H20 的完整 UUID；驱动未装时可暂留空 |
| `max_model_len` | 可调整 | `65536` | 单序列输入+输出上限；提高会增加缓存需求 |
| `max_num_seqs` | 可调整 | `16` | 服务调度上限，非企业用户数；按实测延迟调优 |
| `gpu_memory_utilization` | 可调整 | `0.85` | 每卡显存预算比例，不是权重精度 |
| `release` | 必须确认 | `20261108-01` | 本次发布编号；启动过的发布不可覆盖，调整配置时填写新编号 |
| `tls_mode` | 必须选择 | `enterprise` | 正式用 enterprise；封闭测试可选 self-signed |
| `tls_chain_source` | 企业证书必填 | `/secure-transfer/server-chain.pem` | 企业 CA 返回的服务器完整证书链在本机的路径 |
| `tls_ca_source` | 企业证书必填 | `/secure-transfer/company-ca.pem` | 工作站信任用企业 CA 文件在本机的路径 |
| `alertmanager` | 有告警平台时填写 | `空字符串` | 填主机:端口；留空只采集指标，不向企业通知平台发送告警 |
| `docker_version` | 可保留 | `空字符串` | 首次从官方源选 29.x 并记录；企业有批准版本时填 apt 完整版本字符串 |
| `toolkit_version` | 可保留 | `1.20.1-1` | 填写 APT 完整版本，非只有 1.20.1 |
| `data_root` | 必须确认 | `/srv/phyxi` | 填写已挂载数据卷路径，使用 findmnt 检查；不填已有业务目录 |
| `driver_branch` | 可保留 | `580` | 已有驱动须不低于该分支；安装新驱动需要重启 |
| `model_revision` | 可保留 | `仓库锁定 commit` | 只填写官方 GLM-5.3-Flash 的 40 位 commit；变更后重新下载与验收 |
| `vllm_image` | 可保留 | `仓库锁定 digest` | 填写完整镜像名@sha256:摘要；不可用裸 nightly 代替 |
| `nginx_image` | 可保留 | `nginx:stable-alpine` | 首次解析 digest 固定；企业可填批准的完整 digest |
| `prometheus_image` | 可保留 | `prom/prometheus:v3.5.0` | 首次解析 digest 固定；企业可填批准的完整 digest |
| `model_cpus` | 可调整 | `32` | 模型容器 CPU 配额，按专用主机可用 CPU 确定 |
| `model_memory_gib` | 可调整 | `512` | 模型容器主机内存上限；主机至少多留 64 GiB |
| `shm_gib` | 可调整 | `32` | 容器共享内存预算 |
| `max_num_batched_tokens` | 可调整 | `4096` | 单次调度输入与生成 token 总预算 |
| `max_images` | 可调整 | `4` | 每请求最大图片数；当前不开放视频 |
| `reasoning_effort` | 可选择 | `low` | 只能选 low / high / max；均可能输出思考 token |
| `per_client_connections` | 可调整 | `4` | 网关单工作站 Token 在途连接限制 |
| `total_connections` | 可调整 | `16` | 网关全局在途限制，应不小于单工作站限制 |
| `benchmark_concurrency` | 可调整 | `[1, 2, 4]` | 客户端并发档位；超过单 Token 限制时可能返回 429 |
| `benchmark_requests` | 可调整 | `16` | 每个并发档位的请求总数，预热请求另计 |
| `benchmark_max_tokens` | 可调整 | `1024` | 并发测试每请求输出上限，包含思考 token |
| `verify_max_tokens` | 可调整 | `4096` | 功能验收输出上限；预算太小可能只有思考没有答案 |

**无需自行编造的值：** 后端 Key、工作站 Token 由第 10 节生成；CSR 和 TLS 私钥由第 11 节生成。域名和网段由企业网络管理员确定，证书由企业 CA 签发，不能用随机值代替。

**本方案固定项：** 官方 GLM-5.3-Flash FP8、8 张 H20 / TP8、Python 多模态前端、BF16 KV、glm47 解析器、API 名 `glm-5.3-flash`。这些是方案兼容性基线，不是需要你编辑的脚本参数。

**下载默认使用公共镜像：** 清华 PyPI/Ubuntu/Docker CE、DaoCloud Docker Hub 加速、HF-Mirror、npmmirror。下方 `DOWNLOADS` 可直接修改；详见 `docs/downloads.md`。NVIDIA 软件仓库、Python Windows 安装器和 GitHub 保留官方地址。


In [ ]:
# 读取已有配置，补齐本版本新字段；未填写的字段采用示例默认值。
config = json.loads((REPO / "deployment/config.example.json").read_text())
if CONFIG_PATH.exists():
    config.update(json.loads(CONFIG_PATH.read_text()))

# 直接在本单元把需要修改的 None 替换成实际值。None = 保留当前值。
OVERRIDES = {
    "dedicated_server": None,  # 必须确认：确认是专用服务器；共享训练服务器不使用此 Notebook
    "domain": None,  # 必须填写：向企业网络管理员获取 DNS 域名，解析到 intranet_ip
    "intranet_ip": None,  # 必须填写：用 ip -br addr 查看服务器业务网卡地址
    "allowed_cidrs": None,  # 必须填写：由网络管理员给出工作站/应用出口网段；不是 GPU 网段
    "gpu_uuids": None,  # 必须填写：第 5 节 nvidia-smi 获取八张已分配 H20 的完整 UUID；驱动未装时可暂留空
    "max_model_len": None,  # 可调整：单序列输入+输出上限；提高会增加缓存需求
    "max_num_seqs": None,  # 可调整：服务调度上限，非企业用户数；按实测延迟调优
    "gpu_memory_utilization": None,  # 可调整：每卡显存预算比例，不是权重精度
    "release": None,  # 必须确认：本次发布编号；首次配置生成后不覆盖已有变更版本
    "tls_mode": None,  # 必须选择：正式用 enterprise；封闭测试可选 self-signed
    "tls_chain_source": None,  # 企业证书必填：企业 CA 返回的服务器完整证书链在本机的路径
    "tls_ca_source": None,  # 企业证书必填：工作站信任用企业 CA 文件在本机的路径
    "alertmanager": None,  # 有告警平台时填写：填主机:端口；留空只采集指标，不向企业通知平台发送告警
    "docker_version": None,  # 可保留：首次从官方源选 29.x 并记录；企业有批准版本时填 apt 完整版本字符串
    "toolkit_version": None,  # 可保留：填写 APT 完整版本，非只有 1.20.1
    "data_root": None,  # 必须确认：填写已挂载数据卷路径，使用 findmnt 检查；不填已有业务目录
    "driver_branch": None,  # 可保留：已有驱动须不低于该分支；安装新驱动需要重启
    "model_revision": None,  # 可保留：只填写官方 GLM-5.3-Flash 的 40 位 commit；变更后重新下载与验收
    "vllm_image": None,  # 可保留：填写完整镜像名@sha256:摘要；不可用裸 nightly 代替
    "nginx_image": None,  # 可保留：首次解析 digest 固定；企业可填批准的完整 digest
    "prometheus_image": None,  # 可保留：首次解析 digest 固定；企业可填批准的完整 digest
    "model_cpus": None,  # 可调整：模型容器 CPU 配额，按专用主机可用 CPU 确定
    "model_memory_gib": None,  # 可调整：模型容器主机内存上限；主机至少多留 64 GiB
    "shm_gib": None,  # 可调整：容器共享内存预算
    "max_num_batched_tokens": None,  # 可调整：单次调度输入与生成 token 总预算
    "max_images": None,  # 可调整：每请求最大图片数；当前不开放视频
    "reasoning_effort": None,  # 可选择：只能选 low / high / max；均可能输出思考 token
    "per_client_connections": None,  # 可调整：网关单工作站 Token 在途连接限制
    "total_connections": None,  # 可调整：网关全局在途限制，应不小于单工作站限制
    "benchmark_concurrency": None,  # 可调整：客户端并发档位；超过单 Token 限制时可能返回 429
    "benchmark_requests": None,  # 可调整：每个并发档位的请求总数，预热请求另计
    "benchmark_max_tokens": None,  # 可调整：并发测试每请求输出上限，包含思考 token
    "verify_max_tokens": None,  # 可调整：功能验收输出上限；预算太小可能只有思考没有答案
}
config.update({key: value for key, value in OVERRIDES.items() if value is not None})
import importlib.util
_spec = importlib.util.spec_from_file_location("phyxi_deploy_config", REPO / "scripts/deploy.py")
_deploy = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_deploy)
_deploy.validate(config)  # 仅校验输入，不执行系统操作；此时允许 GPU 列表暂空。
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
CONFIG_PATH.chmod(0o600)
DATA_ROOT = Path(config["data_root"])
print("以下为已保存配置（无密钥），请核对后再执行安装：")
print(json.dumps(config, ensure_ascii=False, indent=2))

# 公共镜像源：直接修改本单元；保存后各部署步骤重新读取，无需修改脚本。
DOWNLOADS = {
    'pip_index_url': 'https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple',
    'ubuntu_mirror': 'https://mirrors.tuna.tsinghua.edu.cn/ubuntu',
    'docker_apt_mirror': 'https://mirrors.tuna.tsinghua.edu.cn/docker-ce/linux/ubuntu',
    'docker_registry_mirrors': ['https://docker.m.daocloud.io'],
    'hf_endpoint': 'https://hf-mirror.com',
    'npm_registry': 'https://registry.npmmirror.com',
    'python_installer_url': 'https://www.python.org/ftp/python/3.12.10/python-3.12.10-amd64.exe',
    'project_archive_url': 'https://github.com/qlsupersky/PhyXiCATIAagent/archive/020ae85544dd55ebbc82166dbb7d3573aa7baee2.zip',
}
(REPO / "downloads.local.json").write_text(json.dumps(DOWNLOADS, ensure_ascii=False, indent=2) + "\n")
print("已保存公共镜像源配置；工作站使用同一 downloads.local.json。")


## 2. 服务器与数据卷检查

检查 Ubuntu 24.04、x86_64、主机内存、独立挂载点与可用空间；拒绝已存在其他容器或 GPU 工作负载的服务器。输出硬件信息。未分配数据卷时先由管理员完成挂载，再重跑本步。
**执行前确定：** `data_root` 的挂载点、磁盘容量和 `domain` 已填妥。可在管理员终端使用 `findmnt`、`lsblk -f`、`df -h`、`ip -br addr` 查看磁盘和网卡。Notebook 不选择磁盘或自动格式化。


In [ ]:
stage("preflight")

## 3. 系统基础依赖与目录

安装证书、网络、Python、内核头文件等组件；创建 `/srv/phyxi/{docker,containerd,models,cache,prometheus,secrets,artifacts,evidence,releases}`，启用 SSH 和时间同步。

In [ ]:
stage("base")

## 4. NVIDIA 驱动

复用已正常工作的 580 或更新驱动；若缺失，安装官方 580 分支驱动。本步安装后主动抛出“需要重启”提示：在管理员终端执行 `reboot`，重启 Jupyter，重新运行 0、1、4 单元。Secure Boot 按提示完成 MOK 注册。不要在 Notebook 中自动重启。
**可在第 1 节修改：** `driver_branch`。默认 580，已有更高且正常工作的驱动直接复用；若企业选其他分支，应先确认它与锁定镜像的 CUDA 兼容。


In [ ]:
stage("driver")

## 5. 记录 GPU UUID

读取 GPU 清单，不加载模型。第一次执行查看清单后，将八张 H20 UUID 填入下方 `GPU_UUIDS` 列表并重新执行此单元，保存到同一个 config.local.json。企业方案使用一份模型跨八卡 TP=8。
**需要填写：** `gpu_uuids`。复制下面的八个完整 UUID 到本节 GPU_UUIDS 列表（也可回到第 1 节填写），不能填 `[0,1,2,3,4,5,6,7]`。配置值只定义设备选择，启动前还会检查它们是否空闲。


In [ ]:
subprocess.run(["nvidia-smi", "--query-gpu=index,uuid,name,memory.total,memory.used", "--format=csv"], check=True)
# 如尚未在第 1 节填写，可直接在这里粘贴八个 UUID；None 保留已保存值。
GPU_UUIDS = None  # 示例：["GPU-完整UUID1", ...]，必须有八个不同 UUID
if GPU_UUIDS is not None:
    config = json.loads(CONFIG_PATH.read_text())
    config["gpu_uuids"] = GPU_UUIDS
    CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
    CONFIG_PATH.chmod(0o600)
assert len(json.loads(CONFIG_PATH.read_text())["gpu_uuids"]) == 8, "在此单元或第 1 节填写八个 UUID 后重跑"


## 6. 安装 Docker 与持久存储

从 Docker 官方仓库选择 29.x，或使用第 1 节指定的批准版本。首次配置镜像与容器存储到 `/srv/phyxi/docker`，containerd 到 `/srv/phyxi/containerd`。记录版本；重跑核对已管理的 Engine；指定新 Docker 版本时先要求部署服务已停止，再安装该版本，不搬迁已有业务数据。此阶段会重启专用服务器的 Docker。
**可在第 1 节修改：** `docker_version`、`data_root`。空版本表示首次自动选择官方源的 Docker 29.x 完整版本，并保存结果。已有批准版本则填完整 APT 字符串。设置过 Engine 存储后不直接修改数据根；数据迁移是独立运维操作。


In [ ]:
stage("docker")

## 7. 安装 GPU 容器运行时

安装固定版本 NVIDIA Container Toolkit，注册 NVIDIA runtime，记录版本并锁定软件包。重启专用服务器 Docker；不加载模型。
**可在第 1 节修改：** `toolkit_version`，例如 `1.20.1-1`。指定值与已安装版本一致时复用；变更时要求先停止部署服务，然后安装指定版本并重启 Docker。


In [ ]:
stage("toolkit")

## 8. 固定镜像与 GLM 兼容性检查

拉取 `deployment/versions.json` 中固定 digest 的 vLLM 镜像；Nginx/Prometheus 首次解析 digest 后保存，后续复用。运行容器检查 GLM5Next 架构、glm47 工具/思考解析器、FlashInfer 最低版本和启动参数；不分配 GPU。结果在 `artifacts/image-compatibility.json`。失败时停止，不能仅凭镜像标签继续。
**可在第 1 节修改：** `vllm_image`、`nginx_image`、`prometheus_image`。默认直接使用仓库批准的输入，不要求你自行寻找新版本。输入不变时复用已锁定摘要；修改任一镜像输入后，要求先停止服务，再重新拉取、锁定并检查。不会静默忽略新输入。


In [ ]:
stage("images")

## 9. 下载固定版本 FP8 权重

下载官方仓库指定 commit；已有同一版本时断点续传。核对 FP8 配置和全部权重分片，逐文件生成 SHA-256。模型目录、总字节数、分片数写入 `artifacts/model.json`。此步磁盘与网络开销较大，不占 GPU。
**可在第 1 节修改：** `model_revision`；默认固定官方 commit 即可。无需修改 `download-model.py`。镜像支持和模型 commit 应配套验收。


In [ ]:
stage("model")

## 10. 生成正式配置与两层密钥

根据第 1 节生成 Compose 的八卡 UUID 列表、模型参数、Nginx 域名与允许网段。建立 `current` 发布软链接；生成后端 API Key、工作站 Token 及网关映射，不打印秘密。配置位于 `releases/<release>/`，凭据位于 `secrets/`；重复运行保留原有凭据。
**本步将配置落到文件：** GPU、资源预算和运行参数 → Compose / `.env`；域名、网段、连接限制 → Nginx；告警地址 → Prometheus。可以在第 13 节查看生成结果。未启动过的发布可重跑本节；启动过的发布必须先停止并在第 1 节填写新 release。程序会生成新目录、保存旧目录位置并切换 current，无需手改脚本或软链接。

**修改后如何生效（仍全部在 Notebook 操作）：**

1. 服务已经运行时，先执行第 19 节停止单元（将 STOP_SERVICE 改为 True）；停止完成后改回 False。
2. 在第 1 节修改参数并保存；若该发布已经启动过，给 release 一个新编号，例如 `20261108-02`。
3. 修改镜像：重跑第 8 节；修改模型 commit：重跑第 9 节；修改 Docker/Toolkit 版本：在服务停止后重跑第 6/7 节。
4. 重跑第 10 节，生成新配置并记录前一发布目录。修改域名时，先重跑第 11 节生成对应 CSR，获得匹配证书后执行第 12 节；证书导入也需在服务停止时执行。
5. 第 13 节查看结果，第 14 节启动，第 15 节回归验收。
6. 仅调整 `benchmark_*`、`verify_max_tokens` 或证书导入源路径，不需要重建模型配置；保存后重新运行对应验收/证书步骤。`reasoning_effort` 同时影响服务默认值，按部署参数处理。

`data_root` 仅支持首次部署选择；部署后更换数据盘属于迁移，不会自动搬迁已有镜像和模型。


In [ ]:
stage("render")

## 11. 生成 TLS 私钥与 CSR

生成服务器私钥和证书签发请求。企业 CA 模式：将 `/srv/phyxi/artifacts/server.csr` 提交 CA；收到完整证书链和根/中间 CA 信任文件后，放到第 1 节配置的 `/secure-transfer/` 路径。私钥始终留在服务器。
**执行前确定：** 第 1 节 `domain` 是最终域名，`tls_mode` 是正式或测试模式。企业证书须使用本步骤生成的 CSR 签发；自行使用另一个私钥签发的证书无法与这里的私钥配对。


In [ ]:
stage("tls_csr")

## 12. 安装并验证 TLS 证书

校验证书域名、链和私钥配对；设置 Nginx 读取权限。企业 CA 文件未就绪则停止；自签名验收模式会生成 30 天证书，并把该证书用作工作站信任锚。
**直接在第 1 节填文件路径：** `tls_chain_source` 与 `tls_ca_source`。复制 CA 文件到服务器后再执行；若为 self-signed，这两个输入路径不读取。私钥和后端 Key 不手工填写到 Notebook。


In [ ]:
stage("tls_install")

## 13. 检查将要执行的配置

本单元只查看配置。关键基线：TP=8，FP8 权重自动按 checkpoint 读取；`dtype=bfloat16` 为未量化计算与 KV 默认精度；`VLLM_USE_RUST_FRONTEND=0`；两种解析器均为 glm47。不是 Qwen 参数。

默认 `MAX_NUM_SEQS=16` 为模型调度初值；Nginx 单工作站在途 4、总在途 16。多个工作站须分别发放 Token。参数解释见 `docs/parameters.md`。
**在第 1 节调节：** 上下文、序列数、显存比例、批 token 预算、图片数、思考强度、CPU/内存/shared memory、网关连接上限。下面显示已生成的发布配置；输入修改后按第 10 节更新。第 14 节会核对输入与发布快照一致，拒绝用旧配置启动。


In [ ]:
print((DATA_ROOT / "current/start-model.sh").read_text())
print((DATA_ROOT / "current/.env").read_text())  # 不包含密钥
subprocess.run(["bash", str(DATA_ROOT / "current/dc.sh"), "config", "--quiet"], check=True)


## 14. 启动服务（本步开始使用八张 GPU）

先核对八卡空闲和 DNS，再运行轻量 CUDA 检查、Nginx 配置检查，安装并启动 `phyxi-model.service`。启动最长等待约 65 分钟，用于首次权重加载和内核制备；网关在模型 healthy 后启动。

失败时在管理员终端查看 `bash /srv/phyxi/current/dc.sh logs --tail 100 model`。不要反复绕过检查，先修复日志中的具体错误。初次安装保持批准的软件源网络可达，完成启动验收后再执行企业出口策略。
**执行前确认：** 本步骤会占用八张 GPU；DNS、证书、入口网络策略和第 13 节参数均已确认。没有额外脚本参数需要修改。


In [ ]:
stage("start")

## 15. TLS、鉴权、图片及多轮工具协议验收

检查错误凭据返回 401、合法模型列表、文本、SSE、红色色块图片、工具调用与第二轮工具回填。默认使用 GLM 的 `reasoning_effort=low`，最多 4,096 输出 token（含思考）；不使用 Qwen 的 enable_thinking 开关。保存 `evidence/api-validation.json`。检查 Prometheus 就绪。
**可在第 1 节修改：** `verify_max_tokens`、`reasoning_effort`。网关 URL、CA、Token 路径和模型名由部署配置自动传入。


In [ ]:
stage("verify")

## 16. 有界并发验收

默认按客户端并发 1、2、4，分别预热一次并执行 16 个单轮请求，最多 1,024 输出 token。完成后保存每请求状态、实际 token、耗时、截断数与可见回答数。TPS 是服务合计 completion_tokens / 测试窗口，包含思考 token、网络与排队；不是单用户文字速度。结果为本机实测，不引用历史 27B 数据。

此组为协议/短负载验收；长期吞吐与多轮建模用户容量需另用项目实际任务集测试。
**可在第 1 节修改：** `benchmark_concurrency`、`benchmark_requests`、`benchmark_max_tokens`、`reasoning_effort`。前三项仅影响验收请求，默认分别为 `[1,2,4]`、16、1024；reasoning_effort 默认为 low，同时影响服务默认值，修改时按第 10 节重新生成部署配置。不需要编辑负载脚本。


In [ ]:
stage("benchmark")

## 17. 导出工作站配置

生成仅管理员可读的 `/srv/phyxi/client-export/`，包含 CA、该工作站 Token、启动脚本和网关地址。不包含后端 Key、TLS 私钥。由企业安全渠道分发给指定工作站；新增工作站需发放不同 Token。

Windows 安装和项目接入执行仓库 `scripts/setup-workstation.ps1`，详见 `docs/workstation.md`。服务端 Notebook 不能代替在 Windows 上安装客户端和 CATIA。项目对话链路已提供版本校验的连接适配；实际 CATIA 工具链是独立业务验收。
**工作站需要确定：** 服务端导出的 endpoint.json 给出 BaseUrl；将 CA、Token 安全分发后，在 Windows 命令中填写这两个本地文件路径。不要把服务器路径直接当作 Windows 路径。


In [ ]:
stage("export")

## 18. 状态与交付记录

确认模型、网关、监控状态，保存软件版本与 GPU 采样。根据 `docs/operations.md` 完成告警送达、证书续签、主机重启恢复和回滚演练，再签署企业交付验收。
**可在第 1 节填写：** `alertmanager`。填写前向企业运维确认主机、端口和通知通道；空值不会发送告警通知。CPU/内存和初始并发阈值由验收数据确认。


In [ ]:
stage("status")
packages = subprocess.check_output(["dpkg-query", "-W", "-f=${Package}\t${Version}\n", "docker-ce", "docker-ce-cli", "containerd.io", "docker-compose-plugin", "nvidia-container-toolkit"], text=True)
(DATA_ROOT / "artifacts/system-packages.tsv").write_text(packages)
print("配置、版本和结果位于：", DATA_ROOT)


## 19. 可选：停止服务并释放 GPU

默认不停止正式服务。需要结束验收时将下面 `STOP_SERVICE` 改为 True。停止 systemd 管理的模型、网关和监控，保留权重、配置、缓存和证据。重启正式服务使用 `systemctl start phyxi-model.service`；启动前检查分配的 GPU 空闲。

In [ ]:
STOP_SERVICE = False
if STOP_SERVICE:
    stage("stop")
else:
    print("未执行停止操作")